In [ ]:
!pip install -U transformers

In [2]:
import os
import json
import torch
import librosa
from tqdm import tqdm
from transformers import AutoProcessor, AutoModelForSpeechSeq2Seq
import soundfile as sf
import warnings
warnings.filterwarnings("ignore")

2025-07-26 08:24:35.021380: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1753518275.261422      36 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1753518275.332540      36 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered


# Load model

In [7]:
# Load model và processor PhoWhisper-large
processor = AutoProcessor.from_pretrained("vinai/PhoWhisper-large")
model = AutoModelForSpeechSeq2Seq.from_pretrained("vinai/PhoWhisper-large")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = model.to(device)

In [4]:
audios_dir = '/kaggle/input/audio-extracted-data/audio_extract'
all_audio_paths = dict()

for part in sorted(os.listdir(audios_dir)):
    all_audio_paths[part] = dict()

for data_part in sorted(all_audio_paths.keys()):
    data_part_path = f'{audios_dir}/{data_part}'
    audio_paths = sorted(os.listdir(data_part_path))
    for audio_path in audio_paths:
        audio_id = audio_path.replace('.wav', '')
        audio_path_full = f'{data_part_path}/{audio_path}'
        all_audio_paths[data_part][audio_id] = audio_path_full

In [5]:
all_audio_paths.keys()

dict_keys(['L21_a', 'L22_a', 'L23_a', 'L24_a', 'L26_a', 'L26_b', 'L26_c', 'L26_d', 'L26_e', 'L27_a', 'L28_a', 'L29_a', 'L30_a'])

# Inference

In [ ]:
save_dir_all = './audio_ARS'
if not os.path.exists(save_dir_all):
    os.mkdir(save_dir_all)

for key in tqdm(all_audio_paths.keys()):
    if key not in ["L21_a", "L22_a", "L23_a", "L24_a"]: # Choose folder to process
        continue
    save_dir = f'{save_dir_all}/{key}'

    if not os.path.exists(save_dir):
        os.mkdir(save_dir)
        
    audio_paths_dict = all_audio_paths[key]
    audio_ids = sorted(audio_paths_dict.keys())
    for audio_id in tqdm(audio_ids):
        audio_path = audio_paths_dict[audio_id]
        
        speech, sampling_rate = librosa.load(audio_path, mono=True, sr=16000)
        speech = speech.astype('float64')
        speech_len = len(speech)
        
        with open(f'/kaggle/input/audio-detect-json/{key}/{audio_id}.json', 'r') as f:
            audio_shots = json.load(f)
        
        results = []
        for audio_shot in audio_shots:
            start, end = audio_shot
            lst_audio_frames = []
            while (end - start) >= 1:
                if (end - start) <= 30: 
                    lst_audio_frames.append(speech[int(start*sampling_rate):min(speech_len, round(end*sampling_rate))])
                    break
                else:
                    lst_audio_frames.append(speech[int(start*sampling_rate):min(speech_len, round((start+30)*sampling_rate))])
                    start += 30

            if lst_audio_frames != []:
                # 1. Chuyển audio thành mel spectrogram
                input_features = processor(
                    lst_audio_frames,
                    sampling_rate=sampling_rate,
                    return_tensors="pt",
                    padding=True  # padding waveform theo batch
                ).input_features.to(device)
            
                # 2. Padding mel spectrogram về 3000 frames nếu cần
                from torch.nn.functional import pad
                if input_features.shape[-1] < 3000:
                    pad_length = 3000 - input_features.shape[-1]
                    input_features = pad(input_features, (0, pad_length), mode='constant', value=0)
            
                # 3. Đảm bảo không vượt quá 3000
                input_features = input_features[:, :, :3000]
            
                # 4. Generate
                generated_ids = model.generate(input_features=input_features)
                result = processor.batch_decode(generated_ids, skip_special_tokens=True)
                result = " ".join(result)
                results.append(result)
            else:
                results.append("")
            
        with open(f'{save_dir}/{audio_id}.json', 'w', encoding='utf-8') as f:
            json.dump(results, f, ensure_ascii=False)

  0%|          | 0/29 [00:00<?, ?it/s]The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.

 10%|█         | 3/29 [19:32<2:48:40, 389.26s/it]